# Hand Gesture Recognition — Pipeline Demo

This notebook demonstrates both the **Classical** (MediaPipe + SVM) and **Deep Learning** (YOLOv8 + ResNet18) pipelines on sample images from the HaGRID test set.

**Classes:** like, dislike, ok, palm, fist, peace


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from glob import glob

from src.evaluation.compare import ClassicalPipeline, DeepPipeline
from src.config import GESTURE_CLASSES

print("Imports OK")
print("Gesture classes:", GESTURE_CLASSES)


In [ ]:
print("Loading Classical Pipeline...")
classical = ClassicalPipeline()
print("Classical Pipeline loaded!")

print("\nLoading Deep Learning Pipeline...")
deep = DeepPipeline()
print("Deep Pipeline loaded!")


In [ ]:
# Pick one sample image per class from the test set
test_dir = Path("data/processed/classifier/test")
sample_images = {}
for cls in GESTURE_CLASSES:
    files = list((test_dir / cls).glob("*.jpg"))
    if files:
        sample_images[cls] = str(files[0])
        
for cls, path in sample_images.items():
    print(f"{cls}: {path}")


In [ ]:
def compare_pipelines(image_path, class_name):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Classical inference
    import time
    t0 = time.perf_counter()
    c_dets = classical.detect(img)
    c_time = (time.perf_counter() - t0) * 1000
    
    # Deep inference
    t0 = time.perf_counter()
    d_dets = deep.detect(img)
    d_time = (time.perf_counter() - t0) * 1000
    
    # Draw classical detections (bbox only)
    c_vis = img_rgb.copy()
    for d in c_dets:
        cls_id, conf, x, y, w, h = d
        cv2.rectangle(c_vis, (x, y), (x+w, y+h), (0, 255, 0), 2)
        label = f"{GESTURE_CLASSES[cls_id]}: {conf:.2f}"
        cv2.putText(c_vis, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    # Draw deep detections
    d_vis = img_rgb.copy()
    for d in d_dets:
        cls_id, conf, x, y, w, h = d
        cv2.rectangle(d_vis, (x, y), (x+w, y+h), (255, 0, 0), 2)
        label = f"{GESTURE_CLASSES[cls_id]}: {conf:.2f}"
        cv2.putText(d_vis, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Original — True: {class_name}")
    axes[0].axis("off")
    
    axes[1].imshow(c_vis)
    c_pred = GESTURE_CLASSES[c_dets[0][0]] if c_dets else "No detection"
    axes[1].set_title(f"Classical: {c_pred}\n({c_time:.1f} ms)")
    axes[1].axis("off")
    
    axes[2].imshow(d_vis)
    d_pred = GESTURE_CLASSES[d_dets[0][0]] if d_dets else "No detection"
    axes[2].set_title(f"Deep: {d_pred}\n({d_time:.1f} ms)")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return {
        "true": class_name,
        "classical_pred": c_pred,
        "deep_pred": d_pred,
        "classical_time_ms": c_time,
        "deep_time_ms": d_time,
    }

print("compare_pipelines() defined")


In [ ]:
results = []
for cls, path in sample_images.items():
    r = compare_pipelines(path, cls)
    results.append(r)
    print(f"{cls}: Classical={r['classical_pred']}, Deep={r['deep_pred']}")


In [ ]:
import pandas as pd
df = pd.DataFrame(results)
print("\n=== Inference Timing Summary ===")
print(f"Classical mean: {df['classical_time_ms'].mean():.1f} ms")
print(f"Deep mean:      {df['deep_time_ms'].mean():.1f} ms")

df


## Try Your Own Image

Upload an image and run both pipelines on it.


In [ ]:
# Uncomment and modify the path below to test on your own image
# my_image_path = "path/to/your/image.jpg"
# compare_pipelines(my_image_path, "unknown")
